# 翌月末MP統合予測モデル
## 当月日次データ → 翌月末MP 2段階予測

### 予測フロー
当月X日時点で、以下の **2段階** で翌月末MPを予測します。

**Stage 1: 当月末までの成長予測（日次データ → 月末値）**
- 今日の次月確定+A、次月B/C/Dの実績値
- → 当月末時点での各値を予測（Random Forest）
- 学習データ：過去の「当月X日 → 同月末日」ペア

**Stage 2: 翌月末MP予測（月末値 → 翌月末MP）**
- Stage 1で予測した月末Kakutei+A（確定部分）
- Stage 1で予測した月末B/C/D（追加部分の変換元）
- → 翌月末MP予測（Random Forest + 転換率ハイブリッド）
- 学習データ：過去の「月末実績 → 翌月末MP」ペア

### 予測例
```
① 2月12日時点 : 次月確定+A = 4,571万円 (実績)
② Stage 1     : 2月末には   5,200万円 (+629万円 成長予測)
③ Stage 2     : 3月末MP   = 6,000万円 (+800万円 BCD転換予測)
```

## 1. ライブラリのインポート

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import calendar
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded!')

## 2. データの読み込み

In [ ]:
try:
    from google.colab import files
    print('Upload CSV file:')
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
except:
    filename = 'progress_data.csv'
    print(f'Using: {filename}')

In [ ]:
try:
    df = pd.read_csv(filename, encoding='utf-8-sig')
except:
    df = pd.read_csv(filename, encoding='shift_jis')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

## 3. データ前処理

In [ ]:
original_columns = df.columns.tolist()
print(f'Original columns: {original_columns}')

df['date'] = pd.to_datetime(df.iloc[:, 0], errors='coerce')
df = df.dropna(subset=['date']).copy()
df['year_month'] = df['date'].dt.to_period('M')
df['month'] = df['date'].dt.month
df['day']   = df['date'].dt.day

if len(original_columns) >= 13:
    col_mapping = {
        original_columns[1]:  'Current_MP',
        original_columns[7]:  'Next_MP',
        original_columns[8]:  'Next_Kakutei',
        original_columns[9]:  'Next_A',
        original_columns[10]: 'Next_B',
        original_columns[11]: 'Next_C',
        original_columns[12]: 'Next_D'
    }
    print('13-column format')
elif len(original_columns) >= 12:
    col_mapping = {
        original_columns[1]:  'Current_MP',
        original_columns[7]:  'Next_Kakutei',
        original_columns[8]:  'Next_A',
        original_columns[9]:  'Next_B',
        original_columns[10]: 'Next_C',
        original_columns[11]: 'Next_D'
    }
    print('12-column format')
else:
    col_mapping = {
        original_columns[1]: 'Current_MP',
        original_columns[2]: 'Next_Kakutei',
        original_columns[3]: 'Next_A',
        original_columns[4]: 'Next_B',
        original_columns[5]: 'Next_C',
        original_columns[6]: 'Next_D'
    }
    print('7-column format')

df = df.rename(columns=col_mapping)

for col in ['Current_MP', 'Next_MP', 'Next_Kakutei', 'Next_A', 'Next_B', 'Next_C', 'Next_D']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

month_end_dates = df.groupby('year_month')['date'].max()
df['is_month_end'] = df.apply(lambda x: x['date'] == month_end_dates[x['year_month']], axis=1)
df['KA'] = df['Next_Kakutei'] + df['Next_A']

print(f'Daily records : {len(df)}')
print(f'Months        : {df["year_month"].nunique()}')
df[['date', 'year_month', 'KA', 'Next_B', 'Next_C', 'Next_D', 'is_month_end']].head()

### 3-1. Stage 1 学習データ作成（日次 → 月末値）
各月の月末値を取得し、月内の各日のレコードに紐付けます。
完全月（月末日まで揃っている月）のみを使用します。

In [ ]:
# 各月の月末値（KA, B, C, D）を取得
me_vals = (
    df[df['is_month_end']]
    .groupby('year_month')[['KA', 'Next_B', 'Next_C', 'Next_D']]
    .last()
    .rename(columns={'KA': 'KA_end', 'Next_B': 'B_end', 'Next_C': 'C_end', 'Next_D': 'D_end'})
)

df['KA_end'] = df['year_month'].map(me_vals['KA_end'])
df['B_end']  = df['year_month'].map(me_vals['B_end'])
df['C_end']  = df['year_month'].map(me_vals['C_end'])
df['D_end']  = df['year_month'].map(me_vals['D_end'])

# 完全月の特定（最終データ日 = 実際のカレンダー月末日）
me_check = df[df['is_month_end']].copy()
me_check['is_actual_end'] = me_check['date'].apply(
    lambda d: d.day == calendar.monthrange(d.year, d.month)[1]
)
complete_months = set(me_check[me_check['is_actual_end']]['year_month'])

# Stage 1 学習データ（月末以外 + 完全月 + ターゲットあり）
growth_train = df[
    ~df['is_month_end'] &
    df['KA_end'].notna() &
    df['year_month'].isin(complete_months)
].copy()

print(f'Stage 1 training data: {len(growth_train)} records, {growth_train["year_month"].nunique()} months')
print(f'Complete months: {len(complete_months)} / {df["year_month"].nunique()} total')
growth_train[['date', 'KA', 'KA_end', 'Next_B', 'B_end']].head()

### 3-2. Stage 2 学習データ作成（月末値 → 翌月末MP）
月末実績と翌月のMP実績をペアリングします。

In [ ]:
# 完全月の月末行のみ抽出
me_df = df[df['is_month_end']].copy()
me_df['is_actual_end'] = me_df['date'].apply(
    lambda d: d.day == calendar.monthrange(d.year, d.month)[1]
)
month_end_df = me_df[me_df['is_actual_end']].sort_values('year_month').reset_index(drop=True)

# 翌月の実績MPとペアリング（連続する月のみ）
month_end_df['Next_Month_MP'] = None
for i in range(len(month_end_df) - 1):
    if month_end_df.loc[i+1, 'year_month'] == month_end_df.loc[i, 'year_month'] + 1:
        month_end_df.loc[i, 'Next_Month_MP'] = month_end_df.loc[i+1, 'Current_MP']

month_end_df['Next_Month_MP'] = pd.to_numeric(month_end_df['Next_Month_MP'], errors='coerce')
month_end_df['source_month'] = month_end_df['year_month'].dt.month
month_end_df['target_month'] = (month_end_df['source_month'] % 12) + 1

monthly_train = month_end_df[month_end_df['Next_Month_MP'].notna()].copy()
monthly_train['Confirmed']       = monthly_train['KA']
monthly_train['BCD_Total']       = monthly_train['Next_B'] + monthly_train['Next_C'] + monthly_train['Next_D']
monthly_train['Additional']      = monthly_train['Next_Month_MP'] - monthly_train['Confirmed']
monthly_train['Conversion_Rate'] = np.where(
    monthly_train['BCD_Total'] > 0,
    monthly_train['Additional'] / monthly_train['BCD_Total'] * 100,
    0
)

print(f'Stage 2 training data: {len(monthly_train)} months')
monthly_train[['year_month', 'KA', 'BCD_Total', 'Next_Month_MP', 'Conversion_Rate']].head()

## 4. Stage 1: 月末成長予測モデル（日次 → 月末値）
当月X日時点の次月データから、当月末時点の次月データを予測します。
- KA (確定+A)、B、C、D それぞれを独立したRandom Forestで予測

In [ ]:
class MonthEndGrowthPredictor:
    """Stage 1: 当月X日の次月データから当月末値を予測"""

    def __init__(self):
        self.ka_model = None
        self.b_model  = None
        self.c_model  = None
        self.d_model  = None

    def _rf(self):
        return RandomForestRegressor(
            n_estimators=150, max_depth=10,
            min_samples_split=5, min_samples_leaf=2, random_state=42
        )

    def train(self, data):
        self.ka_model = self._rf()
        self.ka_model.fit(data[['KA', 'day', 'month']], data['KA_end'])

        self.b_model = self._rf()
        self.b_model.fit(data[['Next_B', 'day', 'month']], data['B_end'])

        self.c_model = self._rf()
        self.c_model.fit(data[['Next_C', 'day', 'month']], data['C_end'])

        self.d_model = self._rf()
        self.d_model.fit(data[['Next_D', 'day', 'month']], data['D_end'])

        metrics = {}
        for name, model, feat, tgt in [
            ('KA+A', self.ka_model, data[['KA',     'day', 'month']], data['KA_end']),
            ('B',    self.b_model,  data[['Next_B', 'day', 'month']], data['B_end']),
            ('C',    self.c_model,  data[['Next_C', 'day', 'month']], data['C_end']),
            ('D',    self.d_model,  data[['Next_D', 'day', 'month']], data['D_end']),
        ]:
            pred = model.predict(feat)
            metrics[name] = {
                'mae': mean_absolute_error(tgt, pred),
                'r2':  r2_score(tgt, pred)
            }
        return metrics

    def predict(self, ka, b, c, d, day, month):
        pred_ka = float(self.ka_model.predict([[ka, day, month]])[0])
        pred_b  = float(self.b_model.predict([[b,  day, month]])[0])
        pred_c  = float(self.c_model.predict([[c,  day, month]])[0])
        pred_d  = float(self.d_model.predict([[d,  day, month]])[0])
        return {
            'pred_ka': pred_ka, 'current_ka': ka, 'ka_growth': pred_ka - ka,
            'pred_b': pred_b, 'pred_c': pred_c, 'pred_d': pred_d,
        }


growth_predictor = MonthEndGrowthPredictor()
m1 = growth_predictor.train(growth_train)

print('Stage 1: Month-End Growth Models')
print('='*52)
for name, v in m1.items():
    mae = v['mae']
    r2  = v['r2']
    print(f'  {name:6s}  MAE: {mae/10000:>8,.0f} (10K JPY)   R2: {r2:.3f}')

## 5. Stage 2: 翌月末MP予測モデル（月末値 → 翌月末MP）
月末時点の次月データから、翌月末のMPを予測します。
- 確定部分（Kakutei+A）+ BCD転換による追加分
- Random Forest (70%) + 月別転換率 (30%) のハイブリッド

In [ ]:
class MonthlyMPPredictor:
    """Stage 2: 月末時点の次月データから翌月末MPを予測"""

    def __init__(self):
        self.model = None
        self.monthly_rates   = {}
        self.overall_avg_rate = None

    def train(self, data):
        for m in range(1, 13):
            sub = data[data['target_month'] == m]
            if len(sub) > 0:
                self.monthly_rates[m] = sub['Conversion_Rate'].mean()
        self.overall_avg_rate = data['Conversion_Rate'].mean()

        X = data[['Next_B', 'Next_C', 'Next_D', 'target_month']]
        y = data['Additional']
        self.model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
        self.model.fit(X, y)

        pred_rf = self.model.predict(X)
        preds = []
        for i, (_, row) in enumerate(data.iterrows()):
            tm   = int(row['target_month'])
            rate = self.monthly_rates.get(tm, self.overall_avg_rate) / 100
            add_rate = row['BCD_Total'] * rate
            preds.append(pred_rf[i] * 0.7 + add_rate * 0.3)

        forecast = data['Confirmed'] + np.array(preds)
        return {
            'mae': mean_absolute_error(data['Next_Month_MP'], forecast),
            'r2':  r2_score(data['Next_Month_MP'], forecast)
        }

    def predict(self, confirmed, b, c, d, target_month):
        bcd    = b + c + d
        add_rf = float(self.model.predict([[b, c, d, target_month]])[0])
        rate   = self.monthly_rates.get(target_month, self.overall_avg_rate) / 100
        add_rate   = bcd * rate
        additional = add_rf * 0.7 + add_rate * 0.3
        return {
            'confirmed':   confirmed,
            'additional':  additional,
            'forecast':    confirmed + additional,
            'bcd':         bcd,
            'rate':        rate * 100,
            'target_month': target_month
        }


monthly_predictor = MonthlyMPPredictor()
m2 = monthly_predictor.train(monthly_train)

print('Stage 2: Monthly MP Model')
print('='*52)
mae2 = m2['mae']
r2_2 = m2['r2']
print(f'  MAE: {mae2/10000:>8,.0f} (10K JPY)')
print(f'  R2:  {r2_2:>8.3f}')
print()

mn = ['','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
print('BCD Conversion Rates by Target Month:')
print('-'*40)
for m in range(1, 13):
    if m in monthly_predictor.monthly_rates:
        rate_m = monthly_predictor.monthly_rates[m]
        print(f'  {m:2d} ({mn[m]}): {rate_m:5.1f}%')

## 6. 統合予測
### 当月X日時点 → 翌月末MP の2段階予測結果

In [ ]:
non_me = df[~df['is_month_end']]
if len(non_me) == 0:
    print('ERROR: No non-month-end data available')
else:
    latest = non_me.iloc[-1]
    month  = int(latest['month'])
    day    = int(latest['day'])
    target_month = (month % 12) + 1

    ka_today = float(latest['KA'])
    b_today  = float(latest['Next_B'])
    c_today  = float(latest['Next_C'])
    d_today  = float(latest['Next_D'])

    # Stage 1: 当月末値を予測
    s1 = growth_predictor.predict(ka_today, b_today, c_today, d_today, day, month)

    # Stage 2: 翌月末MPを予測
    s2 = monthly_predictor.predict(
        confirmed=s1['pred_ka'],
        b=s1['pred_b'], c=s1['pred_c'], d=s1['pred_d'],
        target_month=target_month
    )

    pred_ka_val    = s1['pred_ka']
    ka_growth_val  = s1['ka_growth']
    pred_bcd_val   = s1['pred_b'] + s1['pred_c'] + s1['pred_d']
    confirmed_val  = s2['confirmed']
    additional_val = s2['additional']
    rate_val       = s2['rate']
    forecast_val   = s2['forecast']

    print('='*70)
    print(f'UNIFIED FORECAST:  {mn[month]} Day {day}  ->  {mn[target_month]} Month-End MP')
    print('='*70)
    print()
    print(f'[1] Current Status  ({mn[month]} Day {day})  -- actual values')
    print(f'    Next Kakutei + A  = {ka_today/10000:>10,.0f}  (10K JPY)')
    print(f'    Next B + C + D    = {(b_today+c_today+d_today)/10000:>10,.0f}  (10K JPY)')
    print()
    print(f'[2] Stage 1: Predicted {mn[month]} Month-End Values')
    print(f'    Kakutei + A  ->  {pred_ka_val/10000:>10,.0f}  (10K JPY)   growth: +{ka_growth_val/10000:,.0f}')
    print(f'    B + C + D    ->  {pred_bcd_val/10000:>10,.0f}  (10K JPY)')
    print()
    print(f'[3] Stage 2: {mn[target_month]} Month-End MP Forecast')
    print(f'    Confirmed (predicted month-end Kakutei+A) = {confirmed_val/10000:>10,.0f}  (10K JPY)')
    print(f'    Additional from BCD conversion            = {additional_val/10000:>10,.0f}  (10K JPY)')
    print(f'    BCD Conversion Rate                       = {rate_val:>10.1f}%')
    print()
    print(f'  *** {mn[target_month]} Month-End MP Forecast = {forecast_val/10000:>10,.0f}  (10K JPY) ***')
    print('='*70)

## 7. 日次予測推移グラフ
当月の各日における翌月末MP予測の推移を可視化します。
- 左グラフ：確定部分（Stage 1予測）+ 追加部分（Stage 2予測）の積み上げ
- 右グラフ：今日のKakutei+A実績 vs Stage 1予測の月末値

In [ ]:
latest_ym = df['year_month'].max()
latest_month_data = df[(df['year_month'] == latest_ym) & ~df['is_month_end']]

if len(latest_month_data) == 0:
    print('No non-month-end data available for progression chart')
else:
    month_val = int(latest_month_data.iloc[0]['month'])
    target_m  = (month_val % 12) + 1

    results = []
    for _, row in latest_month_data.iterrows():
        s1 = growth_predictor.predict(
            float(row['KA']),     float(row['Next_B']),
            float(row['Next_C']), float(row['Next_D']),
            int(row['day']), month_val
        )
        s2 = monthly_predictor.predict(
            confirmed=s1['pred_ka'],
            b=s1['pred_b'], c=s1['pred_c'], d=s1['pred_d'],
            target_month=target_m
        )
        results.append({
            'day':         int(row['day']),
            'current_ka':  float(row['KA']),
            'pred_ka_end': s1['pred_ka'],
            'additional':  s2['additional'],
            'forecast':    s2['forecast'],
        })

    sim_df = pd.DataFrame(results)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: total forecast breakdown (stackplot)
    axes[0].stackplot(
        sim_df['day'],
        sim_df['pred_ka_end'] / 10000,
        sim_df['additional']  / 10000,
        labels=['Predicted Kakutei+A at month-end', 'Additional from BCD conversion'],
        colors=['steelblue', 'lightcoral'], alpha=0.8
    )
    axes[0].plot(
        sim_df['day'], sim_df['forecast'] / 10000,
        color='darkred', linewidth=2, marker='o', markersize=4, label='Total Forecast'
    )
    axes[0].set_title(f'Daily Forecast: {latest_ym} -> {mn[target_m]} Month-End MP')
    axes[0].set_xlabel(f'Day of {latest_ym}')
    axes[0].set_ylabel('MP (10K JPY)')
    axes[0].legend(loc='lower right')
    axes[0].grid(True, alpha=0.3)

    # Right: current KA vs predicted month-end KA
    axes[1].plot(
        sim_df['day'], sim_df['current_ka'] / 10000,
        marker='o', linewidth=2, color='lightblue',
        label='Current Kakutei+A (actual)', alpha=0.9
    )
    axes[1].plot(
        sim_df['day'], sim_df['pred_ka_end'] / 10000,
        marker='s', linewidth=2, color='steelblue',
        label='Predicted Month-End Kakutei+A'
    )
    axes[1].fill_between(
        sim_df['day'],
        sim_df['current_ka']  / 10000,
        sim_df['pred_ka_end'] / 10000,
        alpha=0.2, color='steelblue'
    )
    axes[1].set_title('Kakutei+A: Current vs Predicted Month-End')
    axes[1].set_xlabel(f'Day of {latest_ym}')
    axes[1].set_ylabel('Kakutei+A (10K JPY)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f'Daily Forecast Summary for {mn[target_m]} Month-End:')
    print('-'*65)
    print(f'  {"Day":>4}  {"Current KA":>12}  {"Pred ME-KA":>12}  {"Additional":>12}  {"Forecast":>12}')
    print(f'  {"":>4}  {"(10K JPY)":>12}  {"(10K JPY)":>12}  {"(10K JPY)":>12}  {"(10K JPY)":>12}')
    print('-'*65)
    for _, row in sim_df.iterrows():
        d_val   = int(row['day'])
        cur     = int(row['current_ka']  / 10000)
        pred_me = int(row['pred_ka_end'] / 10000)
        add     = int(row['additional']  / 10000)
        fore    = int(row['forecast']    / 10000)
        print(f'  {d_val:>4}  {cur:>12,}  {pred_me:>12,}  {add:>12,}  {fore:>12,}')